In [1]:
%pip install langchain langchain-community langchain-chroma transformers sentence-transformers pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline

# Load PDFs from a folder
def load_docs(folder_path):
    docs = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(folder_path, file))
            docs.extend(loader.load())
    return docs

# Update this path to where your PDFs are stored
docs = load_docs("C:\\Users\\pulki\\Downloads\\Basics_OF_RAG\\data")
print("PDF Pages Loaded:", len(docs))

C:\Users\pulki\AppData\Local\Temp\ipykernel_12380\353504054.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\pulki\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF Pages Loaded: 111


In [3]:
# Split PDFs into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)
chunks = text_splitter.split_documents(docs)
print("Chunks Created:", len(chunks))

Chunks Created: 406


In [4]:
# Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Save texts into Chroma vector DB
texts = [c.page_content for c in chunks]
db = Chroma(
    collection_name="rag_store",
    embedding_function=embedding_model
)
db.add_texts(texts)

# Retriever
retriever = db.as_retriever(search_kwargs={"k": 3})

C:\Users\pulki\AppData\Local\Temp\ipykernel_12380\1081338257.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8793.81it/s]


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def llm_generate(prompt, max_new_tokens=150):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 13377.60it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [16]:
# Agent brain
def agent_controller(query):
    q = query.lower()
    if any(word in q for word in ["pdf", "document", "data", "summarize", "information", "find"]):
        return "search"
    return "direct"

In [17]:
# RAG
def rag_answer(query):
    action = agent_controller(query)

    if action == "search":
        print(f"🕵️ Agent decided to SEARCH document for: '{query}'")

        results = retriever.invoke(query)

        context = "\n".join(
            [r.page_content for r in results]
        )

        final_prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    else:
        print(f"🤖 Agent decided to answer DIRECTLY: '{query}'")

        final_prompt = query

    # Generate answer using FLAN-T5
    response = llm_generate(
        final_prompt,
        max_new_tokens=150
    )

    return response


# Test 1: Document-specific question
query = "Give me a 5-point summary from the PDF"
print(rag_answer(query))

print("-" * 20)

# Test 2: General knowledge question
print(
    rag_answer(
        "What is an Ideal Resume Format? Explain in 50 words."
    )
)

🕵️ Agent decided to SEARCH document for: 'Give me a 5-point summary from the PDF'
The previous chapter gave us an understanding on how a company evolved right from the idea generation stage to all the way till it decides to file for an IPO
--------------------
🤖 Agent decided to answer DIRECTLY: 'What is an Ideal Resume Format? Explain in 50 words.'
An Ideal Resume Format is a resume format that is designed to be used by aspiring professionals. The ideal format is the best way to present yourself to potential employers.


In [18]:
tests = [
    "Answer this question: What is an ideal resume format?",
    
    "Explain what an ideal resume format is in 50 words.",
    
    "Write 5 points explaining the ideal resume format."
]

for prompt in tests:
    print("\nPROMPT:", prompt)
    print("ANSWER:", repr(llm_generate(prompt, 100)))


PROMPT: Answer this question: What is an ideal resume format?
ANSWER: 'An ideal resume format is one in which the information is presented in a chronological order.'

PROMPT: Explain what an ideal resume format is in 50 words.
ANSWER: 'resume format is a resume that outlines your skills and experience.'

PROMPT: Write 5 points explaining the ideal resume format.
ANSWER: 'The ideal format for a resume is one that is readable and easy to read. This format is best suited for those who want to be able to easily read and understand the information on their resume. For example, if you are applying for an MBA, you should have your resume in the format of an MS Word document.'
